In [1]:
import pandas as pd
import numpy as np

#setting the random seed
np.random.seed(42)

#defining scale parameters
n_customers = 5000
n_transactions = 25000

#customer profile setup
print("Generating realistic M-pesa customer demographic profiles...")
customer_ids = [f"CUST_{str(i).zfill(5)}" for i in range(1,n_customers +  1)]

signup_dates = pd.date_range(start="2023-01-01",end = "2025-12-31",periods=n_customers)

#building the customer dataframe
customers_df = pd.DataFrame({
  "customer_id":customer_ids,
  "signup_date":signup_dates,
  "age":np.random.randint(18,70,size = n_customers),
  "region":np.random.choice(["Nairobi","Mombasa","Kisumu","Nakuru","Eldoret"],size=n_customers,p=[0.4,0.2,0.15,0.15,0.1]),
  "kyc_tier":np.random.choice(["Tier_1","Tier_2","Tier_3"],size=n_customers,p=[0.2,0.5,0.3]),
  "is_active_agent_network":np.random.choice([0,1],size=n_customers,p=[0.7,0.3])

})

#transaction logs set up
print("Generating M-pesa transaction logs (peer,paybill,till)...")
tx_customer_ids = np.random.choice(customer_ids,size=n_transactions)
tx_timestamps = pd.date_range(start = "2026-01-01",end="2026-09-26",periods=n_transactions)

#building the transaction dataframe
transactions_df = pd.DataFrame({
    "transaction_id": [f"TXN_{str(i).zfill(7)}" for i in range(1, n_transactions + 1)],
    "customer_id":tx_customer_ids,
    "timestamp":tx_timestamps,
    "amount_kes":np.random.exponential(scale=2500,size=n_transactions).round(2),
    "transaction_type":np.random.choice(["p2p_send","paybill","buy_goods_till","cash_withdrawal"],size=n_transactions,p=[0.45,0.25,0.20,0.10]),
    "fee_kes":lambda df:df['amount_kes'] * 0.015
}) 

transactions_df["fee_kes"] = (transactions_df["amount_kes"] * 0.012).round(2)

#defining churn-the target variable
active_counts = transactions_df.groupby("customer_id").size()
churned_customers = active_counts[active_counts < 3].index.tolist()
customers_df["churn"] = customers_df["customer_id"].isin(churned_customers).astype(int)

#saving and displaying results
customers_df.to_csv("data/raw/customer_demographics.csv",index = False)
transactions_df.to_csv("data/raw/mpesa_transactions.csv",index = False)

print("✅ Phase 1 Data Ingestion Complete!")
print(f"Saved {len(customers_df)} customer records and {len(transactions_df)} transaction logs.")
display(customers_df.head(3))
display(transactions_df.head(3))

Generating realistic M-pesa customer demographic profiles...
Generating M-pesa transaction logs (peer,paybill,till)...
✅ Phase 1 Data Ingestion Complete!
Saved 5000 customer records and 25000 transaction logs.


,customer_id,signup_date,age,region,kyc_tier,is_active_agent_network,churn
0,CUST_00001,2023-01-01 00:00:00.000000,56,Eldoret,Tier_2,0,0
1,CUST_00002,2023-01-01 05:15:25.385077,69,Nairobi,Tier_2,0,1
2,CUST_00003,2023-01-01 10:30:50.770154,46,Nakuru,Tier_3,0,1


,transaction_id,customer_id,timestamp,amount_kes,transaction_type,fee_kes
0,TXN_0000001,CUST_00248,2026-01-01 00:00:00.000000,1823.26,cash_withdrawal,21.88
1,TXN_0000002,CUST_03504,2026-01-01 00:15:26.245049,10353.14,p2p_send,124.24
2,TXN_0000003,CUST_00568,2026-01-01 00:30:52.490099,295.36,p2p_send,3.54
